# Exploración y Análisis de Datos de Demanda

Una vez completada la preparación de los DataFrames, procederemos a su exploración detallada para extraer información clave.

## PRIMERA PARTE

Comenzaremos analizando el archivo [con la evolucion diaria](data_limpio\ev_diaria.csv), que contiene la afluencia diaria de usuarios en el sistema de transporte: **Metro, Cercanías y Autobuses** (EMT y concesiones interurbanas). Nuestro enfoque se centrará en responder a las siguientes preguntas sobre la demanda:

### Análisis Específico de Metro de Madrid
**Volumen de Usuarios:** ¿A cuántos millones asciende el promedio de usuarios diarios del Metro de Madrid?

**Días Extremos:** ¿Cuál fue el día de máximo uso del Metro? ¿Y el día de mínimo uso?

**Patrones de Demanda:** ¿Cómo ha evolucionado la demanda con el tiempo? ¿Está influenciada por el día de la semana?, ¿y por el mes del año?

### Relación con Otros Medios de Transporte
También contextualizaremos los datos de Metro en relación con los otros medios de transporte incluidos en el archivo:

**Cuota:** ¿Qué medio de transporte (Metro, Cercanías o Autobús) registra la mayor afluencia de usuarios?


In [10]:
# Cargamos librerias
import pandas as pd
import plotly.express as px

In [11]:
# Cargamos los datos limpios
ev_diaria = pd.read_csv('../data/data_limpio/ev_diaria.csv', index_col=0)
ev_diaria.tail()

,fecha,dia,mes,año,dia_semana,metro,EMT,conc_carretera,cercanias
1018,2025-10-15,15,10,2025,3,2459845,1726894,1030392,699904
1019,2025-10-16,16,10,2025,4,2566340,1918140,1106096,750967
1020,2025-10-17,17,10,2025,5,2610594,1910265,1115636,734586
1021,2025-10-18,18,10,2025,6,1777134,1066274,536459,436115
1022,2025-10-19,19,10,2025,7,1492078,762593,362184,291338


In [12]:
# hacemos las medias diarias por cada medio de transporte
medias_diarias = ev_diaria[["metro", "EMT", "conc_carretera", "cercanias"]].mean().round(2).to_frame("medias_diarias")
medias_diarias["millones_usuarios"] = (medias_diarias["medias_diarias"]/1000000).round(2)
medias_diarias

,medias_diarias,millones_usuarios
metro,1910648.65,1.91
EMT,1304325.11,1.30
conc_carretera,791992.73,0.79
cercanias,518006.72,0.52


Respondida la primera pregunta **la media diaria de usuarios de metro son 1.91M de usuarios**. 

In [13]:
# día que más se usó el metro
ev_diaria[ev_diaria["metro"] == ev_diaria["metro"].max()]

,fecha,dia,mes,año,dia_semana,metro,EMT,conc_carretera,cercanias
698,2024-11-29,29,11,2024,5,2783341,1516741,966439,728017


In [14]:
# día que menos se usó el metro
ev_diaria[ev_diaria["metro"] == ev_diaria["metro"].min()]

,fecha,dia,mes,año,dia_semana,metro,EMT,conc_carretera,cercanias
358,2023-12-25,25,12,2023,1,669494,360807,186479,188747


### Contextualización
Vamos a generar gráficos para comprobar qué medio de transporte es el más utilizado.

In [15]:
medias_diarias.sort_values(by="millones_usuarios", ascending=False)
medias_diarias.to_excel('../utils/tablas_presentacion/medias_diarias.xlsx')

In [16]:
# ponemos estos datos en un grafico de barras con plotly para visualizar mejor
# Gráfico de barras (usando millones de usuarios)
fig = px.bar(
    medias_diarias,
    x=medias_diarias.index,
    y='millones_usuarios',
    text='millones_usuarios',
    color=medias_diarias.index,
    color_discrete_sequence=px.colors.qualitative.Plotly
)

# Personalizar el diseño
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(
    title='Media diaria de usuarios por modo de transporte (en millones)',
    xaxis_title='Modo de transporte',
    yaxis_title='Millones de usuarios',
    uniformtext_minsize=8,
    uniformtext_mode='hide',
    template='plotly_white'
)

fig.show()

El Metro concentra la mayor demanda diaria de usuarios, superando los 1.9 millones. Los autobuses EMT ocupan el segundo lugar.

## Patrones de demanda
Centrandonos en el metro vamos a averiguar si existen patrones en el uso:

In [17]:
# Vamos a generar un gráfico con la evolución del nº de usuarios diarios a lo largo del tiempo
import plotly.graph_objects as go


# Calcular la media diaria
media_metro = ev_diaria['metro'].mean()

# Crear la figura
fig = go.Figure()

# Línea de usuarios de metro
fig.add_trace(go.Scatter(
    x=ev_diaria['fecha'],
    y=ev_diaria['metro'],
    mode='lines+markers',
    name='Usuarios Metro',
    line=dict(color='blue', width=2)
))

# Línea horizontal de la media
fig.add_trace(go.Scatter(
    x=ev_diaria['fecha'],
    y=[media_metro]*len(ev_diaria),
    mode='lines',
    name=f'Media ({media_metro:.0f})',
    line=dict(color='red', dash='dash')
))

# Personalizar el diseño
fig.update_layout(
    title='Usuarios diarios del Metro',
    xaxis_title='Fecha',
    yaxis_title='Usuarios',
    hovermode='x unified',
    template='plotly_white'
)

fig.show()


In [18]:
media_metro.to_csv('../utils/tablas_presentacion/media_diaria_metro.csv')

AttributeError: 'numpy.float64' object has no attribute 'to_csv'

Parece que hay fluctuaciones tanto dentro de la semana como a lo largo del año. Comenzaremos por estudiar la semana.

In [9]:
# comportamiento de la demanda semanalmente en millones de usuarios
semana = (ev_diaria[["dia_semana","metro","EMT","conc_carretera","cercanias"]].groupby(by= "dia_semana").mean()/1000000).round(2)
semana["dia_semana"] = ["Lunes","Martes","Miercoles","Jueves","Viernes","Sábado","Domingo"]
semana

,metro,EMT,conc_carretera,cercanias,dia_semana
dia_semana,,,,,
1,2.02,1.48,0.91,0.57,Lunes
2,2.15,1.56,0.96,0.60,Martes
3,2.17,1.54,0.95,0.60,Miercoles
4,2.16,1.51,0.94,0.59,Jueves
5,2.16,1.48,0.93,0.58,Viernes
6,1.53,0.90,0.50,0.38,Sábado
7,1.19,0.66,0.35,0.30,Domingo


In [10]:
# Vamos a crear un gráfico de líneas para visualizar la tendencia de uso/índice de cada medio de transporte a lo largo de los días de la semana.
fig = go.Figure()

# Añadir una línea por cada modo de transporte
fig.add_trace(go.Scatter(x=semana['dia_semana'], y=semana['metro'], mode='lines+markers',
                         name='Metro', line=dict(width=3)))
fig.add_trace(go.Scatter(x=semana['dia_semana'], y=semana['EMT'], mode='lines+markers',
                         name='EMT', line=dict(width=3)))
fig.add_trace(go.Scatter(x=semana['dia_semana'], y=semana['conc_carretera'], mode='lines+markers',
                         name='Concesiones carretera', line=dict(width=3)))
fig.add_trace(go.Scatter(x=semana['dia_semana'], y=semana['cercanias'], mode='lines+markers',
                         name='Cercanías', line=dict(width=3)))

# Personalización del gráfico
fig.update_layout(
    title='Usuarios promedio (en millones) por día de la semana y modo de transporte',
    xaxis_title='Día de la semana',
    yaxis_title='Millones de usuarios',
    template='plotly_white',
    hovermode='x unified',
    legend_title='Modo de transporte'
)

fig.show()

Parece que existe una relación entre el día de la semana y el uso de transporte público, siendo los días laborales cuando más se utiliza. Los fines de semana los numeros son menores. 
Procedemos al contraste de hipótesis para probar que existe dicha relación.
Al ser una variable y otra numérica la prueba más adecuada es la de  Mann-Whitney U:

In [11]:
from scipy.stats import pearsonr, chi2_contingency, mannwhitneyu,f_oneway

In [12]:
# Sobre la tabla con todos los datos hacemos una columna nueva con el tipo de día (laboral o fin de semana) y 
# luego hacemos la prueba de Mann-Whitney U
ev_diaria_copy = ev_diaria.copy()
ev_diaria_copy['tipo_dia'] = ev_diaria_copy['dia_semana'].apply(lambda x: 'Laboral' if x in [1, 2, 3, 4, 5] else 'Fin de semana')
ev_diaria_copy.to_excel('../utils/tablas_presentacion/ev_diaria_tipo_dia.xlsx') # saco el excel para la presentación

# Dividimos los datos en dos grupos
grupo_a = ev_diaria_copy.loc[ev_diaria_copy.tipo_dia == "Laboral"]["metro"]
grupo_b = ev_diaria_copy.loc[ev_diaria_copy.tipo_dia == "Fin de semana"]["metro"]
stat, p_value = mannwhitneyu(grupo_a, grupo_b)

# Resultados
print(f'Estadístico U: {stat}, Valor p: {p_value}')



Estadístico U: 196590.0, Valor p: 9.364576882081967e-98


Con un p: 9.364576882081967e-98 podemos descartar la hipótesis nula y llegamos a la primera conclusión.

#### CONCLUSION 1: Los días laborales es cuándo más se utiliza el metro

Seguimos con el estudio de la demanda a traves de los meses. 
Generamos una tabla con la media por mes.

In [13]:
# Tabla con la media por mes transporte
anual = (ev_diaria[["mes","metro","EMT","conc_carretera","cercanias"]].groupby(by= "mes").mean()/1000000).round(2)
anual["mes"] = ["Enero","Febrero","Marzo","Abril","Mayo","Junio","Julio","Agosto","Septiembre","Octubre","Noviembre","Diciembre"]
anual

,metro,EMT,conc_carretera,cercanias,mes
mes,,,,,
1,1.89,1.24,0.76,0.50,Enero
2,2.11,1.39,0.87,0.57,Febrero
3,2.01,1.32,0.82,0.54,Marzo
4,1.98,1.33,0.82,0.53,Abril
5,1.98,1.35,0.82,0.53,Mayo
6,1.97,1.42,0.82,0.54,Junio
7,1.67,1.19,0.69,0.45,Julio
8,1.26,0.89,0.53,0.36,Agosto
9,1.97,1.43,0.86,0.55,Septiembre


In [14]:
# Vamos a crear un gráfico de líneas para visualizar la tendencia de uso/índice de cada medio de transporte a lo largo de los meses del año.
fig = go.Figure()

# Añadir una línea por cada modo de transporte
fig.add_trace(go.Scatter(x=anual['mes'], y=anual['metro'], mode='lines+markers',
                         name='Metro', line=dict(width=3)))
fig.add_trace(go.Scatter(x=anual['mes'], y=anual['EMT'], mode='lines+markers',
                         name='EMT', line=dict(width=3)))
fig.add_trace(go.Scatter(x=anual['mes'], y=anual['conc_carretera'], mode='lines+markers',
                         name='Concesiones carretera', line=dict(width=3)))
fig.add_trace(go.Scatter(x=anual['mes'], y=anual['cercanias'], mode='lines+markers',
                         name='Cercanías', line=dict(width=3)))

# Personalización del gráfico
fig.update_layout(
    title='Usuarios promedio (en millones) por mes y modo de transporte',
    xaxis_title='Mes',
    yaxis_title='Millones de usuarios',
    template='plotly_white',
    hovermode='x unified',
    legend_title='Modo de transporte'
)

fig.show()

Podemos ver que el uso se mantiene durante el año con dos caidas, una más pronunciada en los meses de verano: Julio y Agosto y otra más suave en Diciembre y Enero.
Tenemos una variable categórica y otra numérica con datos que no siguen una distribución normal. Tras una pequeña investigación, se resuelve que la prueba más indicada en este caso es la de Kruskal–Wallis. Es una prueba no paramétrica que evalúa si las medianas de varios grupos son iguales, en este caso, si el uso del metro es estadísticamente diferente entre los meses del año.

In [15]:
ev_diaria_copy_2 = ev_diaria.copy()
ev_diaria_copy_2= ev_diaria_copy_2[['mes','año','metro']]
ev_diaria_copy_2.to_excel('../utils/tablas_presentacion/ev_diaria_mensual_metro.xlsx')

# Realizamos la prueba de Kruskal-Wallis
from scipy.stats import kruskal

groups = [ev_diaria_copy_2[ev_diaria_copy_2['mes'] == month]['metro'] for month in range(1, 13)]
stat, p_value = kruskal(*groups)    

# Resultados
print(f'Estadístico H: {stat}, Valor p: {p_value}')


Estadístico H: 222.31255306812417, Valor p: 1.7024239816227852e-41


Con un p: 1.7024239816227852e-41 podemos descartar la hipótesis nula y llegamos a la segunda conclusión.

#### CONCLUSION 2: Existe relación del mes del año con la demanda del metro, siendo los periodos valle las vacaciones de verano y navidades.

## SEGUNDA PARTE
Pasamos a analizar el [segundo archivo](data_limpio\entradas_historico.csv) que compone el estudio y que limpiamos previamente. En este archivo encontramos la media mensual de entradas por estación de metro.
- En primer lugar, aprovechamos que tenemos los datos desde 2015 para ver la **evolución en la demanda** y si esta crece o decrece.
- En segundo lugar, con la media de entradas por estación, realizaremos un **ranking** con las estaciones más usadas y agrupando los datos por **zona tarifaria** para comprobar si la demanda está o no relaccionada con la zona en la que está la estación.
- Por ultimo, las colocaremos en un plano para comprobar la distribución geográfica de la demanda.


In [16]:
# Cargamos los datos limpios del segundo archivo
entradas_historico = pd.read_csv('../data/data_limpio/entradas_historico.csv', index_col=0)
entradas_historico.tail()

,codigo,nombre,zona,2015-01,2015-02,2015-03,2015-04,2015-05,2015-06,2015-07,...,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09
232,1226,Julián Besteiro,B1,109259.0,99290.0,105204.0,100207.0,103167.0,99360.0,87542.0,...,160309.0,160161,154062,164407,148922,167470,166786,157624,119787,172714
233,1227,Casa del Reloj,B1,88079.0,89719.0,93469.0,90141.0,92631.0,87318.0,75349.0,...,160852.0,152058,149042,158732,139747,157545,151961,136553,123838,162997
234,1228,Hospital Severo Ochoa,B1,64112.0,63182.0,66552.0,63849.0,65593.0,63856.0,55370.0,...,95873.0,101908,102458,109750,96984,107210,102999,91828,69588,106336
235,1229,Leganés Central,B1,137278.0,137175.0,146085.0,143096.0,141403.0,139928.0,124397.0,...,229939.0,226205,238489,252217,222660,232375,226340,210953,160664,251019
236,1230,San Nicasio,B1,79229.0,80923.0,86895.0,82801.0,84451.0,80874.0,74089.0,...,138785.0,139895,142140,147787,131906,144701,141634,129066,95835,155298


In [17]:

# Seleccionar columnas de fechas
fechas = entradas_historico.columns[3:]
entradas_sumadas = entradas_historico[fechas].sum()

# Crear DataFrame
df_evolucion = pd.DataFrame({
    'fecha': fechas,
    'entradas_totales': entradas_sumadas.values
})
df_evolucion['fecha'] = pd.to_datetime(df_evolucion['fecha'], format='%Y-%m')

# Agregar un índice numérico para la regresión
df_evolucion['x_num'] = range(len(df_evolucion))

# Gráfico con línea de tendencia
fig = px.scatter(df_evolucion, x='fecha', y='entradas_totales', trendline='ols',
                 title='Evolución de entradas totales de 2015 a 2025',
                 labels={'fecha': 'Fecha', 'entradas_totales': 'Total de entradas'})

# Mostrar también la línea de evolución original
fig.add_scatter(x=df_evolucion['fecha'], y=df_evolucion['entradas_totales'],
                mode='lines+markers', name='Evolución')

fig.update_layout(template='plotly_white')
fig.show()

#### CONCLUSION 3: Se puede observar cómo afecto la pandemia de COVID-19 a los datos recogidos. (fechas de referencia: inicio de estado de alarma 30 de enero de 2020​ y fecha de fin: 5 de mayo de 2023)

#### CONCLUSION 4: Aún así observamos que la tendencia en el uso es creciente moderado. 

También se puede observar en este gráfico las fluctuaciones de afluencia durante los meses de verano y en navidades.

## Ranking de estaciones
Para no contaminar nuestros datos con las entradas de la época COVID, vamos a coger los datos desde enero de 2023. Elegimos esta fecha porque podemos observar en el gráfico que ya se vuelve a valores pre-covid y además coincide con el dato más antiguo que teníamos en el archivo por día.
Hemos limpiado previamente el archivo y tenemos la [media de entradas](\data_limpio\media_entradas.csv) por estación en el periodo de 2023 a 2025.

In [18]:
# Vamos a ordenar las estaciones por número de entradas medias mensuales en 2023-2025 y crear un ranking
ranking_estaciones = pd.read_csv('../data/data_limpio/media_entradas.csv', index_col=0)
ranking_estaciones = ranking_estaciones.sort_values(by="media_miles", ascending=False).reset_index(drop=True)
ranking_estaciones['ranking'] = ranking_estaciones.index + 1
ranking_estaciones.head(10)


,codigo,nombre,zona,media_miles,ranking
0,311,Moncloa,A,1732.67,1
1,112,Sol,A,1712.89,2
2,625,Príncipe Pío,A,1319.88,3
3,618,Nuevos Ministerios,A,1311.62,4
4,101,Plaza de Castilla,A,1273.37,5
5,411,Avenida de América,A,1012.03,6
6,308,Plaza de España,A,831.35,7
7,116,Atocha-Renfe,A,685.56,8
8,310,Argüelles,A,683.51,9
9,605,Plaza Elíptica,A,670.04,10


In [19]:
ranking_estaciones.to_csv("../utils/tablas_presentacion/ranking_estaciones.csv")

In [20]:
#generamos un grafico de cajas para ver la distribución
fig = px.box(
    ranking_estaciones,
    x='zona',
    y='media_miles',
    color='zona',  # opcional: colorear por zona
    points='all',  # muestra también los puntos individuales
    title='Distribución de media de entradas por zona tarifaria',
    labels={'zona': 'Zona tarifaria', 'media_miles': 'Media de entradas (miles)'}
)

fig.update_layout(template='plotly_white')
fig.show()

In [21]:
#quitamos los outliers para ver mejor la distribución
fig = px.box(
    ranking_estaciones,
    x='zona',
    y='media_miles',
    color='zona',  # opcional: colorear por zona
    points=False,  # sin los puntos individuales
    title='Distribución de media de entradas por zona tarifaria',
    labels={'zona': 'Zona tarifaria', 'media_miles': 'Media de entradas (miles)'}
)

fig.update_layout(template='plotly_white')
fig.show()

Parece que existe una tendencia en los datos en la que las estaciones con mayores entradas se encuentran en la zona A, mientras que las zonas B1 y B2 tienen los datos más bajos.
La distribución de los datos se asemeja a una normal por lo que utilizaremos el test one-way ANOVA para probar la hipótesís 0.

In [22]:
import scipy.stats as stats
# Agrupar por zona
grupos = [g['media_miles'].values for _, g in ranking_estaciones.groupby('zona')]

# ANOVA de una vía
f_stat, p_valor = stats.f_oneway(*grupos)

print(f"F = {f_stat:.3f}, p = {p_valor:.5f}")

F = 5.242, p = 0.00593


Con el p-valor p = 0.00593 queda descartada la hipótesis nula por lo que se puede decir que existe relación entre la zona y las entradas de las estaciones.
#### CONCLUSIÓN 5: La zona tarifaria influye en la demanda. La zona A que corresponde al centro de la ciudad es la que tiene mayor demanda y las zonas mas perifericas tienen menores datos registrados

## TERCERA PARTE
Vamos a desarrollar un plano interactivo en el que se pueda consultar dónde está la estación con su demanda media y el nº que ocupa en el ranking que hemos elaborado.
Para ello vamos a combinar el archivo de [estaciones](\data_limpio\estaciones.csv) ya limpio y el de ranking que acabamos de hacer.

In [23]:
# Cargamos el archivo con la información de las estaciones
estaciones = pd.read_csv('../data/data_limpio/estaciones_unicas.csv', sep=";" ,index_col=0)
estaciones.tail()

,codigo,nombre,CP,poblacion,direccion,zona,correspondencias,latitud,longitud
Column1,,,,,,,,,
284,1229,LEGANES CENTRAL,28913,Leganés,Calle Virgen del Camino N1,B1,12,40.328985,-3.771537
285,1203,ALCORCON CENTRAL,28922,Alcorcón,Avda de Móstoles N12,B1,12,40.350084,-3.831779
286,1206,MOSTOLES CENTRAL,28933,Móstoles,Paseo de la Estación N13,B2,12,40.328500,-3.863542
287,1222,EL CASAR,28903,Getafe,Avda del Casar N2,B1,12,40.318623,-3.709854
288,530,CASA DE CAMPO,28011,Madrid,Ctra del Zoo N1,A,5. 10,40.403242,-3.761014


In [24]:
print(ranking_estaciones.columns)
print(estaciones.columns)

Index(['codigo', 'nombre', 'zona', 'media_miles', 'ranking'], dtype='object')
Index(['codigo', 'nombre', 'CP', 'poblacion', 'direccion', 'zona',
       'correspondencias', 'latitud', 'longitud'],
      dtype='object')


In [25]:
# Incluimos en el archivo de estaciones las columnas de media de entradas y ranking
estaciones_mapa = estaciones.merge(ranking_estaciones[['codigo','media_miles', 'ranking']], on='codigo',how='inner')
estaciones_mapa.head()

,codigo,nombre,CP,poblacion,direccion,zona,correspondencias,latitud,longitud,media_miles,ranking
0,722,ARROYOFRESNO,28035,Madrid,Calle de Federica Montseny N2,A,7,40.490898,-3.726035,26.38,237
1,211,NOVICIADO,28015,Madrid,Calle de San Bernardo N49,A,2. 3. 10,40.424842,-3.707419,163.58,137
2,1013,LAGO,28011,Madrid,Ronda del Lago N3,A,10,40.416414,-3.735631,106.70,194
3,1014,BATAN,28011,Madrid,Paseo de la Venta N1,A,10,40.407860,-3.753110,168.09,132
4,112,SOL,28013,Madrid,Plaza de la Puerta del Sol N6,A,1. 2. 3,40.416876,-3.703262,1712.89,2


In [26]:
estaciones_mapa.to_excel('../utils/tablas_presentacion/estaciones_ranked.xlsx')

In [27]:
# Creamos un mapa base para visualizar las estaciones con su ranking
import folium

# Crear un mapa centrado en Madrid
mapa_madrid = folium.Map(location=[40.4168, -3.7038], zoom_start=11)

# Añadir estaciones al mapa
for _, fila in estaciones_mapa.iterrows():
    folium.CircleMarker(
        location=[fila['latitud'], fila['longitud']],
        radius=6 + fila['media_miles'] / 500,  # tamaño proporcional al tráfico
        color='blue' if fila['zona'] == 'A' else 'green',
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>{fila['nombre']}</b><br>"
            f"Zona: {fila['zona']}<br>"
            f"Ranking: {fila['ranking']}<br>"
            f"Entradas medias: {fila['media_miles']:.0f} miles",
            max_width=250
        )
    ).add_to(mapa_madrid)

# Mostrar el mapa
mapa_madrid

In [ ]:
mapa_madrid.save('mapa_estaciones.html')